# Day 7: 実験設計とmetadata

## この章で解く現実の課題

刺激群と対照群の差を信頼してよいと言うために、どんなmetadataとreplicateが必要かを判断します。

RNA-seq解析では、統計手法より前に実験設計が重要です。条件とバッチが重なっていると、PCAや差次的発現解析で見える差が「刺激の差」なのか「実験日の差」なのか分離できません。


## 最小限の用語

- sample: RNA-seqされた1つの測定対象。
- condition: 比較したい条件。例: control と treated。
- biological replicate: 別々に培養・処理した独立サンプル。
- technical replicate: 同じ生物サンプルを測定だけ繰り返したもの。
- batch: 実験日、試薬ロット、測定laneなど、条件以外で差を生む要因。
- confounding: condition と batch が重なり、どちらの効果か分からない状態。


In [ ]:
import pandas as pd

metadata_good = pd.DataFrame({
    "sample_id": ["C1", "C2", "C3", "T1", "T2", "T3"],
    "condition": ["control", "control", "control", "treated", "treated", "treated"],
    "batch": ["batch1", "batch2", "batch3", "batch1", "batch2", "batch3"],
    "biological_replicate": [1, 2, 3, 1, 2, 3],
})
metadata_good


In [ ]:
# condition と batch の対応表を見る。
# 各batchにcontrolとtreatedが両方入っていれば、条件差とバッチ差を比較的分けやすい。
pd.crosstab(metadata_good["batch"], metadata_good["condition"])


## 悪い設計: 条件とバッチが完全に重なる

次の例では、controlはすべてbatch1、treatedはすべてbatch2です。もしtreated群で発現が高く見えても、それが刺激の効果なのかbatch2の測定条件の効果なのか分かりません。


In [ ]:
metadata_bad = pd.DataFrame({
    "sample_id": ["C1", "C2", "C3", "T1", "T2", "T3"],
    "condition": ["control", "control", "control", "treated", "treated", "treated"],
    "batch": ["batch1", "batch1", "batch1", "batch2", "batch2", "batch2"],
    "biological_replicate": [1, 2, 3, 1, 2, 3],
})
pd.crosstab(metadata_bad["batch"], metadata_bad["condition"])


In [ ]:
def check_confounding(metadata, condition_col="condition", batch_col="batch"):
    table = pd.crosstab(metadata[batch_col], metadata[condition_col])
    # 各batchに複数条件が含まれるか、各conditionが複数batchにまたがるかを確認する。
    batch_has_multiple_conditions = (table > 0).sum(axis=1)
    condition_has_multiple_batches = (table > 0).sum(axis=0)
    return {
        "cross_table": table,
        "min_conditions_per_batch": int(batch_has_multiple_conditions.min()),
        "min_batches_per_condition": int(condition_has_multiple_batches.min()),
        "warning": "condition と batch が重なっている可能性" if int(batch_has_multiple_conditions.min()) == 1 else "大きな完全交絡は見えにくい",
    }

check_confounding(metadata_bad)


## 読み取り

metadataは単なる説明表ではありません。解析可能性を判断するための設計図です。

良いmetadataには、少なくとも `sample_id`, `condition`, `replicate`, `batch` が必要です。公共データを使う場合も、自分の実験データを使う場合も、最初にこの表を作れないデータは危険です。

## エラーが出た場合

- `KeyError: 'batch'`: metadataにbatch列がありません。列名を確認してください。
- `pandas` がない: Colabでは通常利用できます。ローカルではvenvにpandasを入れる必要があります。

## この章のゴール

metadata表を見て、比較してよい設計か、条件とバッチが重なっていないかを判断できれば合格です。
